> **SmartVal AI's problem:** A real-estate analytics startup needs to predict median house values for California districts. Their model must achieve MAE ≤ $40k to pass regulatory approval for automated appraisals. The 8-feature California Housing dataset has 20,640 districts — enough to see overfitting clearly. This notebook builds the complete ML pipeline: linear regression → gradient descent → loss functions → regularisation → classification → overfitting diagnosis.

# ML Basics: Regression, Classification, and Generalisation

| Part | Concept | Key question answered |
|------|---------|----------------------|
| 1 | Linear regression | Predict house price from 8 features; what's our baseline MAE? |
| 2 | Gradient descent from scratch | Can we train a linear model manually and match sklearn? |
| 3 | Loss functions | Does MSE or MAE work better when districts are outliers? |
| 4 | Regularisation | Ridge vs. Lasso: which features matter most? |
| 5 | Classification | Can we classify "high value" districts (>$300k median)? |
| 6 | Overfitting | What happens when we train on only 50 samples? |

---

> **Dataset:** California Housing (`sklearn.datasets.fetch_california_housing`). No download needed. 8 features: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude. Target: median house value ($100k units).

In [ ]:
# ── Dependencies and Data ─────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['numpy','matplotlib','scikit-learn','seaborn']:
    try: __import__(pkg.replace('-','_').split('==')[0])
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, confusion_matrix

np.random.seed(42)

# Load California Housing dataset
housing = fetch_california_housing()
X_raw, y = housing.data, housing.target
feature_names = housing.feature_names

print(f"Dataset: {X_raw.shape[0]:,} districts × {X_raw.shape[1]} features")
print(f"Target: median house value ($100k units)")
print(f"Features: {list(feature_names)}")
print()
print(f"Price range: ${y.min():.2f}00k – ${y.max():.2f}00k")
print(f"Mean price:  ${y.mean():.2f}00k  (≈ ${y.mean()*100:.0f}k)")

In [ ]:
# ── Train/val/test split + feature scaling ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train_s.shape}  |  Val: {X_val_s.shape}  |  Test: {X_test_s.shape}")
print()
print("SmartVal's pipeline:")
print("  1. Standardise features (mean=0, std=1) so all features are comparable")
print("  2. Train on 66% of data, validate on 17%, hold 17% for final evaluation")
print("  3. Never touch the test set until the very end")

---

## Part 1 — Linear Regression: Fitting a Straight Line

Linear regression predicts: $\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_8 x_8 + b$

Each weight $w_i$ says "how much does feature $i$ contribute to the price?" The bias $b$ is a baseline price. We minimise the Mean Squared Error: $\text{MSE} = \frac{1}{n}\sum(\hat{y}_i - y_i)^2$

#### 🔮 Predict first

Before fitting, predict: which single feature will have the highest positive weight in the linear model?

1. **MedInc** (median income) — richer districts have higher house prices
2. **Latitude** — southern California is more expensive than northern
3. **AveRooms** — more rooms = higher price

Make your prediction, then run the fit.

In [ ]:
# ── Part 1: Linear regression ─────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_s, y_train)

train_pred = lr.predict(X_train_s)
val_pred   = lr.predict(X_val_s)

train_mae = mean_absolute_error(y_train, train_pred) * 100  # convert to $k
val_mae   = mean_absolute_error(y_val,   val_pred)   * 100

print(f"Linear regression results:")
print(f"  Train MAE: ${train_mae:.1f}k")
print(f"  Val MAE:   ${val_mae:.1f}k  (target: ≤$40k)")
print()
print("Feature weights (standardised → comparable):")
for name, w in sorted(zip(feature_names, lr.coef_), key=lambda x: abs(x[1]), reverse=True):
    bar = "+" * int(abs(w)*10) if w > 0 else "-" * int(abs(w)*10)
    print(f"  {name:12s}: {w:+.4f}  {bar[:30]}")
print(f"\n  Intercept (bias): {lr.intercept_:.4f}")
print()
print("Prediction check: answer depends on actual run — the model tells you!")

In [ ]:
# ── Part 1: Actual vs. predicted scatter ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_val * 100, val_pred * 100, alpha=0.3, s=15, c='steelblue')
lo, hi = 0, 600
ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual price ($k)'); ax.set_ylabel('Predicted price ($k)')
ax.set_title(f'Linear regression — Val MAE = ${val_mae:.1f}k')
ax.legend(); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
plt.tight_layout(); plt.show()

# District 42 example
d42 = X_val_s[42].reshape(1, -1)
pred_42 = lr.predict(d42)[0] * 100
true_42 = y_val[42] * 100
print(f"SmartVal example — District 42:")
print(f"  Predicted: ${pred_42:.1f}k  |  Actual: ${true_42:.1f}k  |  Error: ${abs(pred_42-true_42):.1f}k")

---

## Part 2 — Gradient Descent from Scratch

Linear regression has a closed-form solution. But most ML models don't. The universal training algorithm is gradient descent:

$$w \leftarrow w - \alpha \cdot \frac{\partial \text{MSE}}{\partial w} = w - \alpha \cdot \frac{2}{n} X^T(Xw - y)$$

Let's implement this manually and verify it converges to the same solution as sklearn.

In [ ]:
# ── Part 2: Linear regression via gradient descent ────────────────────────────
X_gd = np.column_stack([X_train_s, np.ones(len(X_train_s))])  # add bias column
n_features = X_gd.shape[1]

weights = np.zeros(n_features)  # initialise at zero
lr_rate = 0.01
losses = []

print("Manual gradient descent on California Housing:")
for epoch in range(200):
    pred = X_gd @ weights
    residuals = pred - y_train
    mse = (residuals**2).mean()
    losses.append(mse)
    # Gradient of MSE w.r.t. weights: 2/n * X^T (Xw - y)
    grad = 2 / len(y_train) * X_gd.T @ residuals
    weights = weights - lr_rate * grad
    if (epoch + 1) % 50 == 0:
        mae_gd = np.abs(pred - y_train).mean() * 100
        print(f"  epoch {epoch+1:3d}: MSE={mse:.4f}  MAE=${mae_gd:.1f}k")

# Compare to sklearn
sklearn_weights = np.append(lr.coef_, lr.intercept_)
max_diff = np.abs(weights - sklearn_weights).max()
print(f"\n  Max weight difference vs. sklearn: {max_diff:.6f}")
print(f"  ✓ Converged to same solution" if max_diff < 0.01 else f"  → Needs more iterations")

In [ ]:
# ── Part 2: Loss convergence plot ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, 'steelblue', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Gradient descent convergence on California Housing')
plt.tight_layout(); plt.show()
print(f"MSE dropped from {losses[0]:.2f} to {losses[-1]:.4f} in 200 epochs.")
print("→ The SAME gradient descent loop trains every neural network.")

---

## Part 3 — Loss Functions: MSE vs. MAE

MSE ($\sum e_i^2$) penalises large errors heavily — one $1,000k outlier matters more than ten $100k errors. MAE ($\sum |e_i|$) treats all errors equally. Which should SmartVal use?

#### 🔮 Predict first

3 of SmartVal's districts have true values capped at $500k in the dataset (a data artifact causing extreme errors). Will switching from MSE to MAE improve or worsen the overall MAE on normal districts?

1. **Improve** — MAE is less distorted by the $500k outlier districts
2. **Worsen** — MAE-trained models are less precise in general
3. **No change** — both losses lead to the same weights for linear regression

In [ ]:
# ── Part 3: MSE vs. MAE loss comparison ──────────────────────────────────────
from sklearn.linear_model import HuberRegressor

# Find outlier districts (capped at $500k in the dataset)
outlier_mask = y_val >= 4.99  # ≥$499k (the cap artifact)
n_outliers = outlier_mask.sum()
print(f"Districts with price ≥$499k (likely capped): {n_outliers}/{len(y_val)}")

# MSE-trained model (standard sklearn LinearRegression)
mse_val_mae = mean_absolute_error(y_val[~outlier_mask], val_pred[~outlier_mask]) * 100

# Huber regression (robust to outliers, approximates MAE for large errors)
huber = HuberRegressor(epsilon=1.35, max_iter=500)
huber.fit(X_train_s, y_train)
huber_pred = huber.predict(X_val_s)
huber_val_mae = mean_absolute_error(y_val[~outlier_mask], huber_pred[~outlier_mask]) * 100

print(f"\nPerformance on NON-outlier districts:")
print(f"  MSE-trained model MAE:   ${mse_val_mae:.1f}k")
print(f"  Robust model MAE:        ${huber_val_mae:.1f}k")
print()
if huber_val_mae < mse_val_mae:
    print("→ Robust loss improved MAE on normal districts by ignoring outlier distortion")
    print("  Prediction 1 is confirmed.")
else:
    print("→ MSE model performed similarly — outlier count may be too small to matter here")

print()
print("Rule of thumb:")
print("  MSE when you care about large errors (safety-critical predictions)")
print("  MAE or Huber when your dataset has noisy/capped outliers")

---

## Part 4 — Regularisation: Preventing Overfitting and Selecting Features

**Ridge** adds $\lambda \sum w_i^2$ to the loss — all weights shrink toward zero but none reach it.
**Lasso** adds $\lambda \sum |w_i|$ — some weights become exactly zero (feature selection).

For SmartVal, Lasso is useful: if 3 of 8 features are genuinely irrelevant, why pay for that data?

In [ ]:
# ── Part 4: Ridge vs. Lasso regularisation ───────────────────────────────────
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

ridge_maes, lasso_maes = [], []
ridge_nonzero, lasso_nonzero = [], []

for alpha in alphas:
    r = Ridge(alpha=alpha); r.fit(X_train_s, y_train)
    l = Lasso(alpha=alpha, max_iter=5000); l.fit(X_train_s, y_train)

    ridge_maes.append(mean_absolute_error(y_val, r.predict(X_val_s)) * 100)
    lasso_maes.append(mean_absolute_error(y_val, l.predict(X_val_s)) * 100)
    ridge_nonzero.append((r.coef_ != 0).sum())
    lasso_nonzero.append((l.coef_ != 0).sum())

print("α       Ridge MAE   Ridge #feat  Lasso MAE   Lasso #feat")
for a, rm, rf, lm, lf in zip(alphas, ridge_maes, ridge_nonzero, lasso_maes, lasso_nonzero):
    print(f"  {a:.3f}   ${rm:.1f}k     {rf}/8          ${lm:.1f}k     {lf}/8")

# Best regularisation
best_alpha_r = alphas[np.argmin(ridge_maes)]
best_alpha_l = alphas[np.argmin(lasso_maes)]
print(f"\n  Best Ridge α: {best_alpha_r}  |  Best Lasso α: {best_alpha_l}")

# Show which features Lasso zeroes out at best alpha
l_best = Lasso(alpha=best_alpha_l, max_iter=5000); l_best.fit(X_train_s, y_train)
print("\nLasso feature weights at best α:")
for name, w in zip(feature_names, l_best.coef_):
    status = "ZERO" if w == 0 else f"{w:+.4f}"
    print(f"  {name:12s}: {status}")

---

## Part 5 — Binary Classification: "High-Value District?"

SmartVal wants to flag districts where median house value exceeds $300k. This converts the regression problem into binary classification. The model outputs a probability: $\hat{p} = \sigma(w^T x + b)$ where $\sigma$ is the sigmoid function. Training minimises Binary Cross-Entropy loss.

In [ ]:
# ── Part 5: Binary classification ─────────────────────────────────────────────
THRESHOLD = 3.0  # $300k in $100k units
y_train_bin = (y_train >= THRESHOLD).astype(int)
y_val_bin   = (y_val   >= THRESHOLD).astype(int)

clf = LogisticRegression(random_state=42, max_iter=500)
clf.fit(X_train_s, y_train_bin)
val_prob = clf.predict_proba(X_val_s)[:, 1]
val_pred_bin = (val_prob >= 0.5).astype(int)

acc = accuracy_score(y_val_bin, val_pred_bin)
cm  = confusion_matrix(y_val_bin, val_pred_bin)

print(f"High-value district (>${THRESHOLD*100:.0f}k) classification:")
print(f"  Val accuracy: {acc:.1%}")
print()
print("Confusion matrix:")
print(f"          Pred: low  Pred: high")
print(f"  True: low   {cm[0,0]:5d}    {cm[0,1]:5d}")
print(f"  True: high  {cm[1,0]:5d}    {cm[1,1]:5d}")
print()
tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
print(f"  Precision: {precision:.1%}  (of flagged high-value, how many truly are?)")
print(f"  Recall:    {recall:.1%}   (of truly high-value, how many did we catch?)")

In [ ]:
# ── Part 5: Sigmoid function and probability calibration ─────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Sigmoid
z = np.linspace(-6, 6, 200)
sigma = 1 / (1 + np.exp(-z))
ax1.plot(z, sigma, 'steelblue', lw=2)
ax1.axhline(0.5, color='coral', ls='--', lw=1, label='Decision boundary (p=0.5)')
ax1.axvline(0, color='gray', lw=0.5)
ax1.set_xlabel('z = w·x + b'); ax1.set_ylabel('σ(z) = P(high value)')
ax1.set_title('Sigmoid: linear score → probability')
ax1.legend()

# Predicted probabilities distribution
ax2.hist(val_prob[y_val_bin==0], bins=30, alpha=0.6, color='steelblue', label='True: low value')
ax2.hist(val_prob[y_val_bin==1], bins=30, alpha=0.6, color='coral', label='True: high value')
ax2.axvline(0.5, color='black', ls='--', lw=1.5)
ax2.set_xlabel('Predicted P(high value)'); ax2.set_ylabel('Count')
ax2.set_title('Predicted probability distributions')
ax2.legend()

plt.tight_layout(); plt.show()

---

## Part 6 — Overfitting: When the Model Memorises Instead of Learning

SmartVal's marketing team proposes training on just the 50 districts they know best. This is dangerous: a model can memorise 50 data points perfectly but fail completely on new districts.

#### 🔮 Predict first

Training a linear regression on only 50 samples from California Housing, what will the validation MAE be compared to the full 13,000-sample model?

1. **Better** — less data makes the model "focus" on what matters
2. **Worse** — overfitting: the model memorises training noise and fails on new data
3. **Same** — linear regression doesn't overfit regardless of sample size

In [ ]:
# ── Part 6: Overfitting demonstration ────────────────────────────────────────
sample_sizes = [20, 50, 100, 200, 500, 1000, 5000, len(X_train_s)]
train_maes_ov, val_maes_ov = [], []

for n in sample_sizes:
    idx = np.random.choice(len(X_train_s), n, replace=False)
    Xn, yn = X_train_s[idx], y_train[idx]
    m = LinearRegression(); m.fit(Xn, yn)
    train_maes_ov.append(mean_absolute_error(yn, m.predict(Xn)) * 100)
    val_maes_ov.append(mean_absolute_error(y_val, m.predict(X_val_s)) * 100)
    print(f"  n={n:6d}: train MAE=${train_maes_ov[-1]:.1f}k  val MAE=${val_maes_ov[-1]:.1f}k")

print()
print("Prediction check:")
print(f"  n=50 val MAE: ${val_maes_ov[1]:.1f}k  vs.  n=full val MAE: ${val_maes_ov[-1]:.1f}k")
if val_maes_ov[1] > val_maes_ov[-1]:
    print("  → Answer 2 confirmed: fewer samples = worse generalisation (even for linear models)")

In [ ]:
# ── Part 6: Learning curve ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(len(sample_sizes)), train_maes_ov, 'o-', color='steelblue', lw=2, label='Train MAE')
ax.plot(range(len(sample_sizes)), val_maes_ov, 's-', color='coral', lw=2, label='Val MAE')
ax.axhline(40, color='green', ls='--', lw=1.5, label='Target: $40k MAE')
ax.set_xticks(range(len(sample_sizes)))
ax.set_xticklabels([str(n) for n in sample_sizes], rotation=30)
ax.set_xlabel('Training samples'); ax.set_ylabel('MAE ($k)')
ax.set_title('Learning curve: more data → better generalisation')
ax.legend()
plt.tight_layout(); plt.show()

target_n = next((n for n, m in zip(sample_sizes, val_maes_ov) if m <= 40), None)
if target_n:
    print(f"SmartVal needs at least ~{target_n:,} samples to hit the $40k MAE target.")

---

## Summary and Closing Decision

| Part | SmartVal question | Answer |
|------|------------------|--------|
| 1 | Baseline MAE? | Linear regression — run Cell 6 to see val MAE |
| 2 | Can we train manually? | Yes — gradient descent matches sklearn |
| 3 | MSE vs. MAE for outliers? | Huber/robust loss is better when caps exist |
| 4 | Which features matter? | Lasso zeroes out irrelevant ones |
| 5 | Can we classify high-value? | Logistic regression with accuracy/precision/recall printed above |
| 6 | Is 50 samples enough? | No — see learning curve for minimum sample needed |

In [ ]:
# ── Closing Decision — SmartVal's recommendation ──────────────────────────────
print("=" * 55)
print("  CLOSING DECISION — SmartVal AI Model Selection")
print("=" * 55)
print()
print(f"  Baseline linear regression (full dataset):")
print(f"    Val MAE: ${val_mae:.1f}k  {'✓ TARGET MET' if val_mae <= 40 else '✗ Below target'}")
print()
print(f"  High-value district classifier:")
print(f"    Accuracy: {acc:.1%}  |  Precision: {precision:.1%}  |  Recall: {recall:.1%}")
print()
print(f"  Minimum training data for $40k target: ~{target_n or 'more than tested':,} samples")
print()
print("  RECOMMENDATION:")
print("  → Use Ridge regularisation (α=0.1) for the regression model")
print("  → Lasso can reduce feature acquisition costs by zeroing irrelevant features")
print("  → Do NOT train on fewer than 1,000 samples — overfitting will disqualify approval")
print()
print("  NEXT STEP: The linear model assumes a linear relationship between")
print("  features and price. To capture nonlinear patterns (e.g., interaction")
print("  between income AND location), we need neural networks — next chapter.")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- Linear regression — closed-form + gradient descent; matched results
- MSE loss and gradient derivation — implemented from scratch and verified
- Ridge vs. Lasso regularisation — coefficient comparison at 5 α values
- Binary classification — logistic regression, sigmoid, confusion matrix
- Overfitting — learning curves showing training/val divergence

### Tier 2 — Explained but Not Fully Implemented
- **Huber loss** — shown as a robust alternative; loss function explained but custom training loop not built
- **Polynomial features** — mentioned in the "nonlinear patterns" pointer; `sklearn.preprocessing.PolynomialFeatures` exists but not used

### Tier 3 — Named but Out of Scope
- **SVMs** — support vector machines for classification; kernel trick enables nonlinear boundaries; standard in notes/01-ml/02-classification but more advanced than needed here
- **Tree-based models** — Random Forest, Gradient Boosting; strong baselines for tabular data; covered in notes/01-ml/08-ensemble-methods
- **Neural network classifiers** — the natural extension of logistic regression; covered starting in the next chapter

---

## When to Use What — ML Basics

| Situation | Choose | Reason |
|---|---|---|
| Continuous target, linear relationship likely | Linear regression + Ridge | Fast, interpretable, good baseline |
| Continuous target, want feature selection | Lasso | Zeroes out irrelevant features |
| Continuous target, noisy/capped outliers | Huber regression | Robust to extreme values |
| Binary outcome | Logistic regression | Outputs calibrated probability |
| Training data < 1,000 samples | Add regularisation + watch val loss | Risk of overfitting is high |

→ **Next:** `learning/genai-prerequisites/02-neural-networks/` — when the linear relationship assumption breaks (spiral datasets, XOR), we need hidden layers and nonlinear activations.